# 06 Error analysis

Where and why do the models fail? Notebooks 04 and 05 saved the AI score (p(AI)) of every review for every model
in `results/04_models/predictions/`. This notebook reads those files, so **nothing is retrained on a GPU**. Only the
two scikit-learn baselines of notebook 03 are refitted (same settings, deterministic, a few seconds).

1. Which Qwen reviews DistilBERT misses, and how sure it is
2. Style markers: what Mistral, Qwen and humans write differently
3. Concrete failure cases
4. Would another threshold help?
5. How sure are the model rankings? (bootstrap)
6. Stress test: are the models detecting formality instead of authorship?
7. Which real players get flagged (year, length, language)
8. Is length still a giveaway?

Runs on CPU (Colab or locally, about a minute). Tables go to `results/06_error_analysis/`; review text appears only
as a few short quotes, cut to 160 characters and without ids.

## 1. Load the dataset and the saved predictions

In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

# Colab: the team Drive. Locally: the git-ignored files in the repository.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/TA2")
except ImportError:
    BASE = Path("..")
DATA = BASE / "data/final/dataset_v1.csv"
PREDS = BASE / "results/04_models/predictions"
RESULTS = BASE / "results/06_error_analysis"
RESULTS.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(0)

df = pd.read_csv(DATA, dtype={"review_id": str})
df["text"] = df.text.astype(str)

# p(AI) of every saved model, joined by review_id and split
MODELS = {"DistilBERT (cased)": "distilbert", "MLP tfidf (default)": "mlp_lowercase",
          "MLP tfidf (keep_case_punct)": "mlp_case_punct", "CNN": "cnn", "BiLSTM": "bilstm"}
ev = df[df.split != "train"].copy()
for name, short in MODELS.items():
    p = pd.concat([pd.read_csv(f, dtype={"review_id": str}) for f in PREDS.glob(f"{name}__*.csv")])
    ev = ev.merge(p[["review_id", "split", "p_ai"]].rename(columns={"p_ai": "p_" + short}),
                  on=["review_id", "split"], how="left")
ev.groupby("split").size()

split
heldout_qwen      5516
reserve_human    16090
test              2654
val               2652
dtype: int64

## 2. Refit the two baselines of notebook 03

Same features and settings as notebook 03. The check below must reproduce its numbers
(TF-IDF + logistic regression: 99.0% on `test`, 82.8% on `heldout_qwen`).

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

EMOJI = re.compile("[\U0001F300-\U0001FAFF☀-➿⭐❤]")
NO_APOSTROPHE = re.compile(r"\b(dont|cant|wont|didnt|doesnt|isnt|im|ive|youre|thats|its a)\b", re.I)
LOWER_I = re.compile(r"(?:^|\s)i(?:\s|$|[,.!?])")
AI_WORDS = ["immersive", "captivating", "masterpiece", "stunning", "breathtaking", "engaging", "unforgettable",
            "rollercoaster", "testament", "delightful", "seamless", "hooked", "enthusiasts", "gem", "realm"]
AI_WORD_RE = re.compile(r"\b(" + "|".join(AI_WORDS) + r")\b", re.I)

def features(text):
    words = re.findall(r"[A-Za-z']+", text)
    letters = [c for c in text if c.isalpha()]
    s = text.strip()
    return {
        "words": len(text.split()),
        "avg_word_len": np.mean([len(w) for w in words]) if words else 0,
        "ends_with_punct": s[-1:] in list(".!?)…") or bool(EMOJI.search(s[-1:])),
        "exclamations": text.count("!"), "questions": text.count("?"),
        "ellipsis": "..." in text or "…" in text,
        "uppercase_share": sum(c.isupper() for c in letters) / len(letters) if letters else 0,
        "starts_lowercase": s[:1].islower(), "lowercase_i": bool(LOWER_I.search(text)),
        "no_apostrophe": bool(NO_APOSTROPHE.search(text)), "emoji": bool(EMOJI.search(text)),
        "hashtag": "#" in text, "newlines": "\n" in text,
        "ai_words": len(AI_WORD_RE.findall(text)),
        "type_token_ratio": len(set(w.lower() for w in words)) / len(words) if words else 0,
    }

def feature_frame(texts):
    return pd.DataFrame([features(t) for t in texts]).astype(float)

train = df[df.split == "train"]
tfidf = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, lowercase=False,
                    token_pattern=r"[A-Za-z']+|[!?.,…#]+"),
    LogisticRegression(C=4, max_iter=2000)).fit(train.text, train.label)
simple = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(feature_frame(train.text), train.label)
ev["p_tfidf_lr"] = tfidf.predict_proba(ev.text)[:, 1]
ev["p_simple"] = simple.predict_proba(feature_frame(ev.text))[:, 1]

ALL = ["p_distilbert", "p_mlp_lowercase", "p_mlp_case_punct", "p_tfidf_lr", "p_cnn", "p_bilstm"]
test, qwen = ev[ev.split == "test"], ev[ev.split == "heldout_qwen"]
reserve = ev[ev.split == "reserve_human"].copy()
mistral_ai, human_test = test[test.label == 1], test[test.label == 0]
qa = qwen[qwen.label == 1].copy()          # the Qwen reviews (AI side of heldout_qwen)
qa["missed"] = qa.p_distilbert < 0.5

pd.DataFrame({m[2:]: {"test acc": ((test[m] >= .5) == test.label).mean(),
                      "Qwen acc": ((qwen[m] >= .5) == qwen.label).mean(),
                      "Qwen recall": (qa[m] >= .5).mean(),
                      "reserve FPR": (reserve[m] >= .5).mean()} for m in ALL + ["p_simple"]}).T.round(3)

                test acc  Qwen acc  Qwen recall  reserve FPR
distilbert         0.994     0.882        0.771        0.011
mlp_lowercase      0.986     0.869        0.754        0.017
mlp_case_punct     0.993     0.849        0.706        0.007
tfidf_lr           0.990     0.828        0.671        0.011
cnn                0.988     0.819        0.650        0.010
bilstm             0.987     0.784        0.579        0.011
simple             0.916     0.813        0.713        0.119

## 3. Which Qwen reviews does DistilBERT miss?

A review is missed when its AI score is below 0.5. The question is whether the misses are borderline
(then a threshold would help) or confident, and whether the other models miss the same reviews.

In [3]:
summary = pd.Series({
    "Qwen reviews": len(qa),
    "missed by DistilBERT": int(qa.missed.sum()),
    "median AI score of a missed review": round(float(qa[qa.missed].p_distilbert.median()), 4),
    "missed with AI score below 0.05": int((qa.p_distilbert < 0.05).sum()),
    "missed also by TF-IDF + log. regression": int((qa.missed & (qa.p_tfidf_lr < 0.5)).sum()),
    "missed by all six models": int((qa[ALL] < 0.5).all(axis=1).sum()),
})
summary.to_csv(RESULTS / "qwen_misses_summary.csv", header=["value"])
summary

Qwen reviews                               2758.0000
missed by DistilBERT                        632.0000
median AI score of a missed review            0.0004
missed with AI score below 0.05             558.0000
missed also by TF-IDF + log. regression     529.0000
missed by all six models                    330.0000
dtype: float64

In [4]:
qa["target_bucket"] = pd.cut(qa.target_words, [0, 5, 10, 20, 50, 100, 1000]).astype(str)
rows = []
for col in ["prompt_style", "target_bucket", "voted_up", "game", "length_fix"]:
    g = qa.groupby(col).missed.agg(["mean", "size"]).reset_index()
    rows.append(g.rename(columns={col: "value", "mean": "miss_rate", "size": "n"}).assign(group=col))
by_group = pd.concat(rows)[["group", "value", "n", "miss_rate"]].round(3)
by_group.to_csv(RESULTS / "qwen_misses_by_group.csv", index=False)
by_group

            group                        value     n  miss_rate
0    prompt_style                       casual   513      0.222
1    prompt_style                     detailed   606      0.157
2    prompt_style                    emotional   516      0.145
3    prompt_style                     humorous   605      0.364
4    prompt_style                        plain   518      0.247
0   target_bucket                       (0, 5]   313      0.380
1   target_bucket                     (10, 20]   607      0.245
2   target_bucket                  (100, 1000]   283      0.177
3   target_bucket                     (20, 50]   679      0.168
4   target_bucket                      (5, 10]   515      0.291
5   target_bucket                    (50, 100]   361      0.139
0        voted_up                        False   503      0.270
1        voted_up                         True  2255      0.220
0            game             Battlefield 2042   259      0.189
1            game               Cyberpun

## 4. Style markers: what does each writer do?

Share of reviews with each marker (the last row is a mean per review). If a marker is common in Mistral and rare in
Qwen, a detector trained on Mistral will lean on it and miss Qwen.

In [5]:
groups = {"human (test)": human_test, "human (reserve)": reserve, "mistral (test)": mistral_ai,
          "qwen (all)": qa, "qwen missed": qa[qa.missed], "qwen caught": qa[~qa.missed]}
markers = {"starts 'This game'": lambda t: t.str.match(r"^\s*This game"),
           "contains the word 'game'": lambda t: t.str.contains(r"\bgame\b"),
           "contains 'Can't wait'": lambda t: t.str.contains("Can't wait"),
           "starts 'Absolutely'": lambda t: t.str.match(r"^\s*Absolutely"),
           "contains '!'": lambda t: t.str.contains("!"),
           "AI words per review": lambda t: t.map(lambda x: len(AI_WORD_RE.findall(x)))}
style = pd.DataFrame({m: {g: f(d.text).mean() for g, d in groups.items()} for m, f in markers.items()}).T.round(3)
style.to_csv(RESULTS / "style_markers.csv")
style

                          human (test)  human (reserve)  mistral (test)  qwen (all)  qwen missed  qwen caught
starts 'This game'               0.048            0.046           0.000       0.132        0.411        0.048
contains the word 'game'         0.610            0.612           0.216       0.339        0.641        0.249
contains 'Can't wait'            0.002            0.003           0.120       0.002        0.000        0.002
starts 'Absolutely'              0.003            0.004           0.222       0.240        0.027        0.303
contains '!'                     0.139            0.160           0.770       0.640        0.495        0.683
AI words per review              0.042            0.061           0.845       0.323        0.128        0.381

In [6]:
# The same in the TF-IDF model: weight of a few tokens (positive = towards AI)
vec, lr = tfidf.named_steps["tfidfvectorizer"], tfidf.named_steps["logisticregression"]
vocab = vec.vocabulary_
pd.Series({w: round(lr.coef_[0][vocab[w]], 2) for w in ["game", "This", "This game", "Can't", "Absolutely", "!"] if w in vocab})

game         -12.23
This          -2.40
This game     -1.27
Can't          4.67
Absolutely     6.76
!             16.74
dtype: float64

## 5. Concrete failure cases

The Qwen reviews DistilBERT is surest are human, and the real players it is surest are AI.
Short quotes only (160 characters), no ids.

In [7]:
cases = pd.concat([
    qa.nsmallest(10, "p_distilbert").assign(case="Qwen review missed by DistilBERT"),
    reserve.nlargest(10, "p_distilbert").assign(case="Real player flagged by DistilBERT"),
])
cases["text"] = cases.text.str.slice(0, 160)
cases = cases[["case", "game", "voted_up", "prompt_style", "p_distilbert", "p_tfidf_lr", "text"]].round(3)
cases.to_csv(RESULTS / "failure_cases.csv", index=False)
for r in cases.itertuples():
    print(f"{r.case[:20]:20} | {r.p_distilbert:.3f} | {r.game[:22]:22} | {r.text[:100]!r}")

Qwen review missed b | 0.000 | Hollow Knight          | 'This game is overpriced in my opinion. The story could have been much more intriguing with better ch'
Qwen review missed b | 0.000 | Sid Meier's Civilizati | 'This game is like having history as your personal plaything!'
Qwen review missed b | 0.000 | Stardew Valley         | 'This game is so relaxing and fun!'
Qwen review missed b | 0.000 | The Witcher 3: Wild Hu | 'This game? Worth every hair on your head and then some!'
Qwen review missed b | 0.000 | Cyberpunk 2077         | "This game is so bad, it's good!"
Qwen review missed b | 0.000 | Slay the Spire         | 'Finally found a game that makes me feel like a real card wizard instead of just a "spend money to ge'
Qwen review missed b | 0.000 | PUBG: Battlegrounds    | 'This game is sick, super fun to play with friends!'
Qwen review missed b | 0.000 | Stardew Valley         | 'This game is super fun with its farming and fishing mini-games.'
Qwen review missed b | 0.000 | Cyber

## 6. Would another threshold help?

ROC-AUC on Qwen is 0.97 but recall only 77%. If the missed reviews scored just below 0.5, a lower threshold would
recover them. We choose the threshold on validation humans (1% flagged) and look at Qwen recall and the reserve FPR.

In [8]:
val_h = ev[(ev.split == "val") & (ev.label == 0)]
rows = []
for m in ["p_distilbert", "p_mlp_lowercase", "p_tfidf_lr"]:
    for name, thr in [("0.5", 0.5), ("1% FPR on validation humans", float(np.quantile(val_h[m], 0.99)))]:
        rows.append({"model": m[2:], "threshold": name, "value": round(thr, 4),
                     "qwen_recall": (qa[m] >= thr).mean(), "reserve_fpr": (reserve[m] >= thr).mean()})
thresholds = pd.DataFrame(rows).round(4)
thresholds.to_csv(RESULTS / "threshold_check.csv", index=False)
thresholds

           model                    threshold   value  qwen_recall  reserve_fpr
0     distilbert                          0.5  0.5000       0.7708       0.0112
1     distilbert  1% FPR on validation humans  0.1274       0.7872       0.0143
2  mlp_lowercase                          0.5  0.5000       0.7538       0.0170
3  mlp_lowercase  1% FPR on validation humans  0.6267       0.7194       0.0114
4       tfidf_lr                          0.5  0.5000       0.6711       0.0113
5       tfidf_lr  1% FPR on validation humans  0.5814       0.6135       0.0080

## 7. How sure are the rankings?

Bootstrap (2,000 resamples) of the test sets. This covers only which reviews happen to be in the test set,
not training randomness: each model was trained once.

In [9]:
rows = []
for split, d in [("test", test), ("heldout_qwen", qwen)]:
    correct = {m: ((d[m] >= 0.5) == d.label).values.astype(int) for m in ALL}
    idx = rng.integers(0, len(d), (2000, len(d)))
    for m in ALL:
        b = correct[m][idx].mean(1)
        rows.append({"split": split, "comparison": m[2:], "accuracy": correct[m].mean(),
                     "ci_low": np.quantile(b, .025), "ci_high": np.quantile(b, .975)})
    for a, b2 in [("p_distilbert", "p_mlp_lowercase"), ("p_distilbert", "p_mlp_case_punct"),
                  ("p_distilbert", "p_tfidf_lr"), ("p_mlp_lowercase", "p_mlp_case_punct")]:
        diff = (correct[a][idx] - correct[b2][idx]).mean(1)
        rows.append({"split": split, "comparison": f"{a[2:]} minus {b2[2:]}",
                     "accuracy": correct[a].mean() - correct[b2].mean(),
                     "ci_low": np.quantile(diff, .025), "ci_high": np.quantile(diff, .975)})
ci = pd.DataFrame(rows).round(4)
ci.to_csv(RESULTS / "bootstrap_ci.csv", index=False)
ci

           split                          comparison  accuracy  ci_low  ci_high
0           test                          distilbert    0.9940  0.9906   0.9966
1           test                       mlp_lowercase    0.9857  0.9808   0.9902
2           test                      mlp_case_punct    0.9928  0.9894   0.9959
3           test                            tfidf_lr    0.9898  0.9857   0.9936
4           test                                 cnn    0.9876  0.9830   0.9917
5           test                              bilstm    0.9872  0.9827   0.9913
6           test      distilbert minus mlp_lowercase    0.0083  0.0034   0.0132
7           test     distilbert minus mlp_case_punct    0.0011 -0.0026   0.0049
8           test           distilbert minus tfidf_lr    0.0041  0.0000   0.0083
9           test  mlp_lowercase minus mlp_case_punct   -0.0072 -0.0121  -0.0026
10  heldout_qwen                          distilbert    0.8816  0.8731   0.8898
11  heldout_qwen                       m

## 8. Stress test: formality or authorship?

Humans write lowercase starts, "i" and no apostrophes; the models never do. If the detectors only detect that,
four mechanical edits should fool them. AI reviews are made **sloppy**, human reviews **polished**.
DistilBERT would need a GPU, so this uses the two refitted baselines.

In [10]:
def sloppy(t):
    t = t.strip()
    t = re.sub(r"\bI\b", "i", t)
    t = t.replace("'", "").replace("’", "")
    t = re.sub(r"[.!?…]+$", "", t)
    return t[:1].lower() + t[1:]

def polish(t):
    t = t.strip()
    t = re.sub(r"(?<![A-Za-z'])i(?![A-Za-z'])", "I", t)
    t = t[:1].upper() + t[1:]
    return t if re.search(r"[.!?…]$", t) else t + "."

print(sloppy("I loved it, can't stop playing!"), "|", polish("its ok i guess"))
rows = []
for name, d, f in [("Qwen AI", qa, sloppy), ("Mistral test AI", mistral_ai, sloppy), ("Human reserve", reserve, polish)]:
    edited = [f(t) for t in d.text]
    rows.append({"reviews": name, "edit": f.__name__,
                 "tfidf_lr_before": (d.p_tfidf_lr >= .5).mean(),
                 "tfidf_lr_after": (tfidf.predict_proba(edited)[:, 1] >= .5).mean(),
                 "simple_before": (d.p_simple >= .5).mean(),
                 "simple_after": (simple.predict_proba(feature_frame(edited))[:, 1] >= .5).mean()})
stress = pd.DataFrame(rows).round(3)
stress.to_csv(RESULTS / "stress_test.csv", index=False)
stress

i loved it, cant stop playing | Its ok I guess.


           reviews    edit  tfidf_lr_before  tfidf_lr_after  simple_before  simple_after
0          Qwen AI  sloppy            0.671           0.477          0.713         0.000
1  Mistral test AI  sloppy            0.991           0.944          0.916         0.001
2    Human reserve  polish            0.011           0.016          0.119         0.195

## 9. Which real players get flagged?

False-positive rate on the human-only reserve by group. **Year** tests the time confound (human reviews 2015–2022,
AI reviews 2026): if the model learned "old language = human", recent reviews would be flagged more.
**English words** is a rough language check: the share of very common English words in the review.

In [11]:
SW = set("the a an and or but is are was were be been it this that to of in on for with as at by from i you he "
         "she we they my your not no do does did have has had so very just game".split())
def english_share(t):
    w = re.findall(r"[A-Za-z']+", t.lower())
    return np.mean([x in SW for x in w]) if w else 0.0

reserve["length"] = pd.cut(reserve.word_count, [0, 5, 10, 20, 50, 100, 300, 5000]).astype(str)
share = reserve.text.map(english_share)
reserve["english_words"] = np.where(share < 0.10, "few (<10% common English words)",
                                    np.where(share < 0.20, "10-20%", "normal (>=20%)"))
rows = []
for col in ["year", "length", "game", "voted_up", "english_words"]:
    g = reserve.groupby(col).agg(n=("p_distilbert", "size"),
                                 fpr_distilbert=("p_distilbert", lambda x: (x >= .5).mean()),
                                 fpr_tfidf_lr=("p_tfidf_lr", lambda x: (x >= .5).mean()),
                                 fpr_mlp_lowercase=("p_mlp_lowercase", lambda x: (x >= .5).mean())).reset_index()
    rows.append(g.rename(columns={col: "value"}).assign(group=col))
fpr = pd.concat(rows)[["group", "value", "n", "fpr_distilbert", "fpr_tfidf_lr", "fpr_mlp_lowercase"]].round(4)
fpr.to_csv(RESULTS / "reserve_fpr_by_group.csv", index=False)
fpr[fpr.group.isin(["year", "length", "english_words"])]

           group                            value      n  fpr_distilbert  fpr_tfidf_lr  fpr_mlp_lowercase
0           year                           2015.0    480          0.0062        0.0146             0.0042
1           year                           2016.0    820          0.0110        0.0073             0.0220
2           year                           2017.0   1573          0.0102        0.0146             0.0153
3           year                           2018.0   1811          0.0088        0.0094             0.0110
4           year                           2019.0   1652          0.0109        0.0097             0.0145
5           year                           2020.0   2447          0.0131        0.0102             0.0155
6           year                           2021.0   3365          0.0107        0.0131             0.0235
7           year                           2022.0   3942          0.0127        0.0112             0.0173
0         length                           (0,

## 10. Is length still a giveaway?

The length-only baseline of notebook 03 is linear (longer = AI). A non-linear model on the word count can also
use "very short targets come out too long".

In [12]:
from sklearn.ensemble import HistGradientBoostingClassifier
n_words = lambda d: d.text.str.split().str.len().to_frame("words")
gbm = HistGradientBoostingClassifier(random_state=0).fit(n_words(train), train.label)
length_check = pd.DataFrame([{"split": s, "accuracy": ((gbm.predict_proba(n_words(d))[:, 1] >= .5) == d.label).mean()}
                             for s, d in [("test", test), ("heldout_qwen", qwen)]]).round(3)
length_check.to_csv(RESULTS / "length_only_nonlinear.csv", index=False)
length_check

          split  accuracy
0          test     0.563
1  heldout_qwen     0.507

## Findings (26 Nov 2026)

- **DistilBERT's Qwen misses are confident, not borderline:** 632 missed, median AI score 0.000, 558 below 0.05.
  A threshold set for 1% FPR on validation humans (0.127) raises Qwen recall only from 77.1% to 78.7%.
  The other models miss the same reviews (330 are missed by all six).
- **"This game" reads as human:** Mistral never starts a review with "This game", so every training review that does
  is human. 41.1% of the missed Qwen reviews start this way (13.2% of all Qwen, 4.8% of humans). The detectors learned
  Mistral's habits ("Can't wait…" 12.0% Mistral vs 0.2% Qwen), not AI writing in general.
- **Misses depend on prompt and length:** humorous 36.4% vs emotional 14.5%; targets of ≤ 5 words 38.0% vs 51–100 words 13.9%.
- **Formality, not authorship:** four mechanical edits lower TF-IDF + logistic regression's Qwen recall from 67.1% to 47.7%;
  the simple-features model drops to 0.0%.
- **Who gets flagged:** short reviews (≤ 5 words: 1.93% vs 0.32% at 100–300 words) and reviews with few common English
  words (3.1% vs 0.9%). The year makes no difference (flat 2015–2022), so we found no sign of the time confound on the human side.
- **Rankings:** on Mistral the best models are within noise; on Qwen DistilBERT leads the lowercase MLP by 1.25 points
  [0.4, 2.1], but each model was trained once, so we treat them as roughly tied.
- **Length:** a non-linear word-count model reaches 56.3% on test (dummy 50%): a weak giveaway, not a strong one.